# 深層学習(Deep Neural Networks)


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/4/4.ipynb)

* 入力データ
* 出力データ（正解）

入力データと出力データの組を**学習**する

入力データから出力データを再現できるような**関数**を作ることができるものを**機械学習**という。

関数はパラメータを持つ。線形回帰なら、パラメータ（傾きと切片）を調節して、予測データと正解データの「誤差」が
最小となるようなパラメータを求める。(「誤差」の評価はスケール変数とカテゴリ変数で異なる。)

深層ニューラルネットワークは隠れ層を入れることでパラメータの数を増やして、近似的な**関数**を作る仕組みである。

生物のニューロンにおいて、刺激の総和が閾値を超えると発火するという概念をまねたものを人工ニューロン（パーセプトロン）という。

単層パーセプトロン(SLP)：入力と出力しかなく、入力の重み付き和が閾値をこえるかどうかが出力である。線形分類（直線や平面でデータが完全に分けられる）しかできない。
$$
f(x_1, x_2) = w_1 x_1 + w_2 x_2 + b > 0 
$$
<img src='SLP.png' width='10%'>

出力に階段関数の代わりにsigmoid関数を使うと、前回のロジスティック回帰と同じになる（多クラスならsoftmax関数）。つまり、ロジスティック回帰は中間層のないニューラルネットワークである。


多層パーセプトロン(MLP)：隠れ層（中間層）を入れたもの。層を増やすことでパラメータを増やすことと、
非線形な活性化関数（例：シグモイド関数、階段関数、ReLU、softmax）を使うことで非線形な分類ができるようになる。

$$
a^{(1)} = g(W^{(1)} a^{(0)} + b^{(1)}),\quad a^{(2)} = g(W^{(2)} a^{(1)} + b^{(2)})
$$
$a^{(k)}$: 第k層の値（ベクトル）、$W$: 重み（行列）、$b$: バイアス、$g$: 活性化関数
<img src='MLP.png' width='20%'>

ニューラルネットワーク(NN)はより一般的な計算モデルで、MLPもその1つ。深層学習はDeep Neural Networkで複数の中間層を意味する。

[様々なNN](https://www.asimovinstitute.org/wp-content/uploads/2019/04/NeuralNetworkZo19High.png)

## 誤差逆伝播法（バックプロパゲーション）
重みパラメータ$w$とバイアス$b$を調整して、入力から求めた予測値と正解データとの誤差が小さくするよう最適化して$(w,b)$を決定する。
よいモデルであれば、どんな入力に対しても正解データに近い予測が得られるはず。

考え方は第1回の練習問題の勾配降下法と同じで、誤差をパラメータで微分した勾配を求め、勾配と逆方向にパラメータを少しずつ動かす。層が多いと微分の計算が大変になるが、出力層から入力層の方向へ順に（逆向きに）計算すると効率よく求められる。これが誤差逆伝播法である。

In [ ]:
# 活性化関数
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def step(z):
    return np.heaviside(z, 0.5)
def ReLU(z):
    return np.maximum(0, z)

x = np.linspace(-10, 10, 200)
y1 = sigmoid(x)
y2 = step(x)
y3 = ReLU(x)
plt.figure(figsize=(4,2))
plt.plot(x, y1, label='sigmoid')
plt.plot(x, y2, label='step')
plt.plot(x, y3, label='ReLU')
plt.legend()
plt.ylim(0,1.1)
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
# アヤメデータの読み込み
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()
print(iris.data[:2])
print(iris.data.shape)
print(iris.target)

In [ ]:
from sklearn import preprocessing
# データの正規化(平均0、分散1)
# 本来は訓練データだけで平均・標準偏差を求め、検証データにも同じ変換をする（検証データの情報が学習に漏れないようにするため）
# アヤメでは結果はほとんど変わらないので、ここでは簡単のためデータ全体で正規化してから分割する
scaler = preprocessing.StandardScaler()
scaler.fit(iris.data)
#x = iris.data
x = scaler.transform(iris.data)
print(x[:2])
print(x.mean(), x.std())

## ラベルの形
前回のロジスティック回帰と同じく、ラベル（正解）は 0, 1, 2 の**整数のまま**使う。

（参考）one-hot 表現：深層学習ではカテゴリー変数を 0 → [1,0,0]、1 → [0,1,0]、2 → [0,0,1] のように変換して使うこともある（統計ソフトではダミー変数という）。この形にした場合は、下の誤差関数を `CategoricalCrossentropy()` にする。この授業では使わない。

In [ ]:
t = iris.target # ラベルは 0, 1, 2 の整数のまま
print(t[[49, 99, 149]])

# （参考）one-hot 表現に変換する場合
#from tensorflow.keras.utils import to_categorical
#print(to_categorical(t)[[49, 99, 149]])

<img src="split.png" width="30%">

In [ ]:
# 訓練用と検証用に分割
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, t, test_size=0.3, random_state=0) 
#x_train, x_test, y_train, y_test = train_test_split(x, t, test_size=0.3, random_state=0, stratify=t) # データセットの比率に合わせて分割（層化サンプリング）
#stratify=t で各カテゴリの比率が同程度になるように分割

## ニューラルネットワーク

* 入力　4次元ベクトル（アヤメの特徴量）
* 出力　3次元ベクトル（アヤメの種類の確率）
* 中間層　M次元ベクトル（N層）M,Nの値は任意（ハイパーパラメータ）

## 使用するライブラリ
* scikit-learn 訓練用と検証用の分割、前処理などに使う
* tensorflow 直接表には出てこないが、多次元配列の計算がGPUで高速化される
* keras ニューラルネットワークを扱う
* (pytorch というライブラリもあるが、ここでは扱わない)

`model = Sequential([])`

の中に層の種類と大きさ、活性化関数の種類を指定する。
* `Input()` 入力層
* `Dense()` 全結合層

2次元画像に有用なCNNなどの発展形では更に増える

誤差関数 loss の取り方（分類の場合）

* 正解ラベルが整数（0, 1, 2, ...）：`SparseCategoricalCrossentropy()`　← この授業ではこれを使う（2値でも多クラスでも同じ）
* 正解ラベルが one-hot（[1,0,0], ...）：`CategoricalCrossentropy()`
* 回帰問題（数値の予測）も扱えるがここでは略

出力層はクラス数 K 個の `Dense(K, activation='softmax')` にする。アヤメなら K=3、MNIST なら K=10、2値分類なら K=2 である。

（参考）2値分類では、出力層を `Dense(1, activation='sigmoid')`、誤差関数を `BinaryCrossentropy()` にする書き方もある。中間層がなければ前回のロジスティック回帰と同じ形になる。

<img src='dnn.png' width=50%>

In [ ]:
# この例では中間層はN=2層, どちらも大きさはM=6
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.utils import set_random_seed

set_random_seed(0) # 重みの初期値や学習データの順番に乱数が使われるので、乱数を固定して毎回同じ結果にする
model = Sequential([
    Input(shape=(4,)), # 入力層　（x_trainの形状に対応させる）
    Dense(6, activation='relu'), # 中間層
    Dense(6, activation='relu'), # 中間層
    Dense(3, activation='softmax') # 出力層 (クラス数 K=3。各種である確率)
])

model.compile(optimizer='adam', loss=SparseCategoricalCrossentropy(), metrics=['accuracy'])
model.summary()

In [ ]:
# 訓練 epochs, batch_size はハイパーパラメータ
# 1epochあたり 全サンプル数/バッチサイズ の回数だけ重みが更新される
history = model.fit(x_train, y_train, epochs=200, batch_size=32, validation_data=(x_test, y_test)) 

In [ ]:
# 学習の様子を確認する　loss(誤差)はできるだけ小さく、accuracy(正解率)はできるだけ大きくなるのがよい
# val_ がついているのは検証データでの値。訓練データの値だけよくなり、検証データの値が悪くなっていく場合は過学習
import matplotlib.pyplot as plt
import pandas as pd
pd.DataFrame(history.history).plot(subplots=[('loss', 'val_loss'), ('accuracy', 'val_accuracy')], layout=(1, 2), figsize=(10, 4), xlabel='Epoch')
plt.show()

In [ ]:
# 訓練後の評価
loss, accuracy = model.evaluate(x_test, y_test)
print(loss, accuracy)

In [ ]:
# 予測
# model.predict は各クラスの確率を返す（前回の model.predict_proba に相当。各行の合計は1）
prob = model.predict(x_test)
print(prob[:5].round(3))

# 確率が最大のクラスを予測とする（前回の model.predict に相当）
y_pred = np.argmax(prob, axis=1) # axis=1: 各行（1件ごと）で最大値の位置（クラス番号）を求める
print('予測', y_pred[:20])
print('正解', y_test[:20])

In [ ]:
# 混同行列と分類レポート（前回と同じ方法）
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=iris.target_names)
plt.show()

### 前回のロジスティック回帰と比べる
同じ訓練データ・検証データでロジスティック回帰を行い、正解率を比べてみよ。

アヤメのように特徴量が少なく単純なデータでは、ニューラルネットワークにしても正解率はほとんど変わらない。ニューラルネットワークが力を発揮するのは、画像のように特徴量（画素）が多いデータである（→ mnist.ipynb）。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)
lr.fit(x_train, y_train)
print('正解率 ロジスティック回帰　:', accuracy_score(y_test, lr.predict(x_test)))
print('正解率 ニューラルネットワーク:', accuracy_score(y_test, y_pred))

In [ ]:
# 前回の練習問題にあったpenguinデータセットで同様の分類ができる。
# 前回ロジスティック回帰で扱った乳がんデータ(sklearn.datasets.load_breast_cancer)でもできる（前回と同様に悪性を1に変換する）。
#   2値分類はクラス数 K=2 の分類として、出力層を Dense(2, activation='softmax') にすればよい（誤差関数は同じ）
#   悪性である確率は model.predict(x_test)[:, 1]（前回の model.predict_proba(X_test)[:, 1] に相当）
# この確率から、前回と同じように roc_auc_score でAUCを求めて、ロジスティック回帰と比べてみよ。